In [1]:
# Movie dataset (features for content-based)
movies = {
    "Movie1": {"Action":1, "Romance":0, "Comedy":0},
    "Movie2": {"Action":1, "Romance":1, "Comedy":0},
    "Movie3": {"Action":0, "Romance":1, "Comedy":1},
    "Movie4": {"Action":0, "Romance":0, "Comedy":1}
}

# User ratings (for CF & SVD)
ratings = {
    "User1": {"Movie1":5, "Movie2":3, "Movie3":0, "Movie4":1},
    "User2": {"Movie1":4, "Movie2":0, "Movie3":4, "Movie4":1},
    "User3": {"Movie1":1, "Movie2":1, "Movie3":0, "Movie4":5},
    "User4": {"Movie1":0, "Movie2":1, "Movie3":5, "Movie4":4}
}

In [2]:
# Step 1: Build user profile (average weighted features)
def build_profile(user):
    profile = {"Action":0, "Romance":0, "Comedy":0}
    total = 0
    
    for movie, rating in ratings[user].items():
        if rating > 0:
            for f in profile:
                profile[f] += rating * movies[movie][f]
            total += rating
    
    for f in profile:
        profile[f] /= total
    
    return profile

# Step 2: Score movies
def score_movies(user):
    profile = build_profile(user)
    scores = {}
    
    for movie in movies:
        if ratings[user][movie] == 0:  # not rated
            score = 0
            for f in profile:
                score += profile[f] * movies[movie][f]
            scores[movie] = score
    
    return scores

print("Content-Based:", score_movies("User1"))

Content-Based: {'Movie3': 0.4444444444444444}


In [3]:
import math

# Cosine similarity between two movies
def cosine(m1, m2):
    num, d1, d2 = 0, 0, 0
    
    for user in ratings:
        r1 = ratings[user][m1]
        r2 = ratings[user][m2]
        
        num += r1 * r2
        d1 += r1**2
        d2 += r2**2
    
    if d1 == 0 or d2 == 0:
        return 0
    
    return num / (math.sqrt(d1) * math.sqrt(d2))

# Predict rating
def predict(user, target_movie):
    num, den = 0, 0
    
    for movie in movies:
        if ratings[user][movie] > 0 and movie != target_movie:
            sim = cosine(target_movie, movie)
            num += sim * ratings[user][movie]
            den += abs(sim)
    
    return num / den if den != 0 else 0

print("CF Prediction:", predict("User1", "Movie3"))

CF Prediction: 2.6880425775294587


In [4]:
import random

users = list(ratings.keys())
items = list(movies.keys())

# Initialize latent matrices
k = 2  # latent factors

U = {u: [random.random() for _ in range(k)] for u in users}
V = {i: [random.random() for _ in range(k)] for i in items}

# Training (gradient descent)
alpha = 0.01
epochs = 500

for _ in range(epochs):
    for u in users:
        for i in items:
            r = ratings[u][i]
            if r > 0:
                pred = sum(U[u][f] * V[i][f] for f in range(k))
                e = r - pred
                
                for f in range(k):
                    U[u][f] += alpha * e * V[i][f]
                    V[i][f] += alpha * e * U[u][f]

# Prediction
def predict_svd(user, movie):
    return sum(U[user][f] * V[movie][f] for f in range(k))

print("Latent Factor Prediction:", predict_svd("User1", "Movie3"))

Latent Factor Prediction: 4.809391848181778
